# Chapter 9: Self-Supervised Learning

This notebook accompanies **Chapter 9** of the lecture notes.

> Last lecture the encoder arrived pretrained, someone else paid for it, and the question was how to reuse it. Today there is no encoder. The corpus is a body of raw text — Schiller's *Die Räuber*, in this case — and the only supervision available is whatever signal we can extract from the data itself. Mask part of the input and predict it; mask a span and predict the span; mask the future and predict the next token. The same principle, three different masking patterns. The representation comes for free.

**Agenda**

🔡 · 🎭 · 🧩 · 📜 · 🗣️ · 🏁

**Take it from here:** 🪞

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Training the tiny model in section 🎭 takes about a minute in this kernel (the gradient is finite-differenced, so each L-BFGS-B step touches every parameter); the same architecture is reused (with different masking) in 🧩 and 📜. The "next scale up" weights produced by `generate_precomputed.py` are loadable via the optional cell beneath each training step.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_attention, check_mask_tokens, check_masked_token_loss,
    check_geometric_span_mask, check_causal_mask, check_next_token_loss,
    check_temperature_sample, check_ema_update, check_jepa_loss,
)
from viz_helpers import (
    load_corpus, build_char_vocab, encode_text, decode_ids, visible_vocab,
    init_params, forward, train_lm, sample,
    load_precomputed_weights, precomputed_meta,
    plot_loss_curve, plot_attention_masks, show_fill_in_examples,
    show_span_vs_single, plot_jepa_variance,
    DEFAULT_D_MODEL, DEFAULT_CONTEXT, MASK_ID,
)

RNG = np.random.default_rng(0)


## 🔡 Corpus & Tokenisation

A *corpus* is the raw thing: a body of text that nobody has labelled, that nobody assembled with a downstream task in mind. The whole point of self-supervised learning is to take that body of text and extract supervision from its own structure.

We use a single play, Friedrich Schiller's *Die Räuber* (1781), as our corpus. Around 277,000 characters of German Sturm und Drang in plain text, public domain via Project Gutenberg. One play is small for a real language model — internet-scale runs use trillions of tokens — but it is large enough that a tiny transformer can extract real structure from it, and small enough that we can fit it in memory and train on the whole thing at once.

> Before we can mask anything we have to decide what counts as a unit. What is the trade-off between modelling characters one at a time and grouping them into subwords or whole words?

<details><summary>Thought</summary>

Smaller units mean more steps per sentence and a longer effective context for the same number of input tokens, but the model has to learn the morphology from scratch — *Räuber* is six tokens at the character level, one token if we had a word vocabulary that included it. Larger units front-load the morphology into the tokeniser and shorten the sequence, but introduce out-of-vocabulary failures and an extra step (the tokeniser) that has to be trained or hand-built. For a small notebook on a single play, character-level keeps the vocabulary tiny (~85 entries) and means we can implement everything in two lines, and that is what we do here.
</details>


In [ ]:
text = load_corpus('die_raeuber.txt')
print(f'Corpus length : {len(text):,} chars')
print(f'First 200 chars:')
print(text[:200])


### Build a char-level vocab

We map every distinct character to an integer id. Slot 0 is reserved for a `[MASK]` sentinel — a token that does not exist in the corpus and that we will use later to hide positions during masked-LM training. Real characters start at id 1.


In [ ]:
stoi, itos = build_char_vocab(text)
vocab_size = len(stoi)
print(f'Vocab size: {vocab_size}  (slot 0 = [MASK])')
print()
print('First 30 vocab entries:')
print(visible_vocab(stoi, n=30))


### Encode the play

`encode_text` turns the string into a long array of integer ids; `decode_ids` reverses it. Round-tripping a sentence is the cheapest sanity check that the tokeniser works.


In [ ]:
ids = encode_text(text, stoi)
print(f'Encoded shape : {ids.shape}, dtype {ids.dtype}')
print(f'First 60 ids  : {ids[:60].tolist()}')
print(f'Round-trip    : {decode_ids(ids[:60], itos)!r}')


**Observe:**
- The integer sequence is exactly as long as the original text — char-level means one token per character.
- Vocab size is small (~85). At a real scale (BPE on web text) it would be 32k–128k, but the *mechanism* is identical: a fixed vocabulary mapped to integer ids.
- Index 0 is empty in the encoded text; we keep it free for the `[MASK]` token we will introduce in the next section.

**At scale.** `mrkschtr/tiny_schiller` on Hugging Face uses a subword tokeniser (BPE-style) trained on a much larger Schiller corpus. The vocabulary ends up around 8k tokens, and a phrase like *„Karl Moor"* might be 2–3 subword tokens rather than 9 characters. The shape of the rest of the pipeline is the same: encode → forward → predict → decode.


## 🎭 Masked-Token Prediction (BERT)

The simplest pretext task on text is the cloze: cover up a token, ask the model to fill it in. The network has to develop an internal sense of which characters are likely in any given context, and the only label needed is the original token, which the data itself provides. This is BERT's recipe at its core, applied here at character resolution.

A masked-LM is naturally **bidirectional**: when predicting a masked position, the model is allowed to see characters both before and after it. The attention pattern is unrestricted — every position can look at every other position, including itself.


### Bidirectional attention

The attention mechanism is the centrepiece of every transformer in this notebook. Three tensors `Q`, `K`, `V`, all with shape `(B, T, D)`, are derived from the input. The output at position `i` is a weighted sum of the value vectors, where the weights come from a softmax over the dot products between query `i` and every key.

> Why is the dot product divided by `sqrt(D)` before softmax? What goes wrong if you skip that step?

<details><summary>Thought</summary>

For random unit vectors in `D` dimensions the dot product has variance roughly `D`. Without dividing by `sqrt(D)` the scores grow with the embedding dimension, the softmax saturates (one token gets all the weight, every other gets zero), and the gradient through that softmax collapses. Dividing by `sqrt(D)` keeps the scores in a regime where the softmax has gradient on more than one entry; it is the only modification of "raw" dot-product attention in scaled dot-product attention.
</details>

Implement `attention(Q, K, V, attn_mask=None)`. It should compute scaled dot-product attention. When `attn_mask` is `None`, every position attends to every other (the bidirectional case). When `attn_mask` is provided, it is an additive mask of shape `(T, T)` — `0` for allowed cells and `-inf` for blocked cells — added to the scores before the softmax.

Useful: `np.swapaxes(K, -1, -2)` to transpose the last two axes; `np.exp`, `.sum(axis=-1, keepdims=True)` for softmax; subtract the per-row max before exponentiating for numerical stability.


In [ ]:
def attention(Q, K, V, attn_mask=None):
    """Scaled dot-product attention.

    Parameters
    ----------
    Q, K, V    : ndarray of shape (B, T, D)
    attn_mask  : ndarray of shape (T, T), additive (0 allowed, -inf blocked), or None.

    Returns
    -------
    ndarray of shape (B, T, D)
    """
    # 1. scores = Q @ K.T / sqrt(D)            (broadcast over batch)
    # 2. if attn_mask is not None:  scores = scores + attn_mask
    # 3. weights = softmax(scores, axis=-1)    (subtract max for stability)
    # 4. return weights @ V
    # YOUR CODE HERE
    pass


check_attention(attention)


### Mask 15% of the tokens — BERT's recipe

The original BERT paper masks 15% of the input tokens. Of those *chosen* positions, 80% are replaced with the `[MASK]` token, 10% are replaced with a *random* token from the vocabulary, and 10% are *kept unchanged*. The mixture matters: training only with `[MASK]` would create a distribution gap between training (where the symbol appears) and inference (where it does not).

Implement `mask_tokens(ids, mask_id, vocab_size, frac, rng)`. Return a tuple `(corrupted_ids, mask_positions)` where `mask_positions` is a boolean array of shape `ids.shape` marking the positions chosen for prediction (True at the 15%, False elsewhere).

Useful: `rng.random(shape) < frac` for boolean selection; `rng.integers(low, high, size)` for random tokens; `np.where` to assemble the corrupted output.


In [ ]:
def mask_tokens(ids, mask_id, vocab_size, frac, rng):
    """Apply BERT-style 80/10/10 masking. ids has shape (B, T)."""
    # 1. chosen = rng.random(ids.shape) < frac           # bool (B, T)
    # 2. sub    = rng.random(ids.shape)                  # for 80/10/10 split
    # 3. is_mask  = chosen & (sub < 0.8)
    #    is_rand  = chosen & (sub >= 0.8) & (sub < 0.9)
    # 4. corrupted = ids.copy()
    #    corrupted[is_mask] = mask_id
    #    rand_ids = rng.integers(1, vocab_size, ids.shape)   # avoid mask_id (0)
    #    corrupted[is_rand] = rand_ids[is_rand]
    # 5. return corrupted, chosen
    # YOUR CODE HERE
    pass


check_mask_tokens(mask_tokens)


### Cross-entropy at the masked positions only

The loss is cross-entropy, but only at the positions we chose to mask. Other positions are not predicted (they were not corrupted), so they should not contribute to the gradient.

> The model produces logits for *every* position. Why is it important to compute the loss at the masked positions only, and not the cross-entropy averaged over all `B*T` positions?

<details><summary>Thought</summary>

Two reasons. The first is mechanical: only at masked positions is the model actually being asked to predict something it does not already see. At non-masked positions the input *contains* the answer, so the model can copy it through the residual stream — the loss there is zero by construction once the model has learned the trivial copy, and contributes no useful gradient. The second is interpretive: averaging over all positions dilutes the signal at masked positions by a factor of `1/frac` (so ~7× at 15% masking), which both slows learning and confuses metrics — the per-token loss of an unmasked position is meaningless.
</details>

Implement `masked_token_loss(logits, targets, mask_positions)`. Use a numerically stable log-softmax (subtract the max across the vocab axis before exponentiating).


In [ ]:
def masked_token_loss(logits, targets, mask_positions):
    """Cross-entropy at positions where mask_positions is True.

    Parameters
    ----------
    logits         : (B, T, V)
    targets        : (B, T) int
    mask_positions : (B, T) bool
    """
    # 1. m = logits.max(axis=-1, keepdims=True)
    # 2. log_probs = logits - m - log(sum(exp(logits - m), axis=-1, keepdims=True))
    # 3. flat_logp = log_probs[mask_positions]      # (n_masked, V)
    #    flat_tgt  = targets[mask_positions]        # (n_masked,)
    # 4. return -mean(flat_logp[arange(n_masked), flat_tgt])
    # YOUR CODE HERE
    pass


check_masked_token_loss(masked_token_loss)


### Train

Helper `train_lm` calls into your `attention` and `masked_token_loss` for the forward pass and the loss; it wraps `scipy.optimize.minimize` for the actual gradient steps. The default tiny model (1 attention block, `d_model=16`, context 16, ~5k parameters) trains in roughly 30–60 seconds in this kernel.

If you would rather skip the training step, the cell below it loads weights pretrained offline at a slightly larger scale (`d_model=64`, 2 layers, 1500 steps).


In [ ]:
# Train the tiny masked LM yourself (default).
params_bd = init_params(vocab_size, seed=0)

def _mask_strategy(batch, rng):
    return mask_tokens(batch, MASK_ID, vocab_size, frac=0.15, rng=rng)

if attention(np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32)) is None:
    print('⬜ Implement attention above first.')
elif masked_token_loss(np.zeros((1, 1, vocab_size), dtype=np.float32),
                       np.zeros((1, 1), dtype=np.int64),
                       np.array([[True]])) is None:
    print('⬜ Implement masked_token_loss above first.')
else:
    params_bd, losses_bd = train_lm(
        params_bd, ids, attention,
        mask_strategy_fn=_mask_strategy,
        use_causal_mask=False,
        n_steps=30, batch_size=32, seed=0,
    )
    plot_loss_curve(losses_bd, 'Bidirectional masked LM training',
                    baseline=float(np.log(vocab_size)))


In [ ]:
# OPTIONAL: replace the in-browser-trained weights with the precomputed ones.
# These were trained offline by generate_precomputed.py at d_model=32 (4× wider
# than the tiny in-browser model) for 1500 Adam steps on the full corpus.
# Same architecture (single block, single head), so the same `forward` works.
#
# Set USE_PRECOMPUTED = True to swap them in for the demos below.

USE_PRECOMPUTED = False
if USE_PRECOMPUTED:
    pre = load_precomputed_weights('bidirectional_lm_weights.npz')
    if pre is None:
        print('No precomputed weights found. Run generate_precomputed.py first.')
    else:
        meta = precomputed_meta('bidirectional_lm_weights.npz')
        print(f'Using precomputed weights:  d_model={meta["d_model"]}  '
              f'context={meta["context"]}  layers={meta["n_layers"]}')
        params_bd = pre


### Fill-in-the-blank

The model has been trained to denoise corrupted Schiller. Pick a few random windows from the play, mask 15% of them with the same strategy, and let the model fill in.


In [ ]:
if attention(np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32)) is None:
    print('⬜ Need attention implemented to run the demo.')
else:
    show_fill_in_examples(params_bd, ids, itos, attention, _mask_strategy,
                          n_examples=4, seed=2)


**Observe:**
- After 40 L-BFGS-B steps the model is nowhere near perfect, but the predictions at masked positions are no longer uniform — common German bigrams (`ch`, `en`, `ie`, `er`) come back; the rare characters and the morphological agreement are out of reach for a 5k-parameter model.
- The reconstruction is *bidirectional*: at every masked position, the model sees both left and right context. Try predicting these from the left side only and the task gets noticeably harder.
- BERT's 80/10/10 mixture matters at scale; in our tiny setting it mostly prevents the model from latching onto the `[MASK]` token id as a shortcut.


## 🧩 Span Masking (SpanBERT, T5)

Single-token masking is often too easy: with twenty visible neighbours, the model can interpolate the missing character from the local n-gram statistics. The harder pretext task is **span masking**: mask a *contiguous chunk* of several tokens at once, so that the model has to reconstruct multi-token structure from the surrounding context.

> What is the model forced to learn from span masking that it cannot learn from single-token masking, even at the same overall mask fraction?

<details><summary>Thought</summary>

Single-token masking can be solved by short-range copying from the immediate neighbours — *yellow [?] peel* is almost always *banana* given the two visible words. Span masking removes a contiguous block — *yellow [? ? ?] and fell over* — so the model has to reconstruct the whole phrase together: it cannot copy the missing words from their direct neighbours because those neighbours are also missing. The model is forced to integrate longer-range context (the rest of the sentence) and to model the *joint* distribution of the masked tokens, not just their marginals one at a time.
</details>

Implement `geometric_span_mask(ids, mask_id, mean_span, mask_frac, rng)`. Pick span starts uniformly at random; for each start, sample a span length from a geometric distribution with mean `mean_span`; mask the contiguous run by replacing those positions with `mask_id`. Stop when roughly `mask_frac * T` positions have been masked. Return `(corrupted_ids, mask_positions)`.

Useful: `rng.geometric(p, size)` returns geometric samples with parameter `p = 1/mean`; clip the span end so it does not exceed `T`; check overlap with already-masked positions and skip on collision.


In [ ]:
def geometric_span_mask(ids, mask_id, mean_span, mask_frac, rng):
    """Span-mask each row of ids. Returns (corrupted, mask_positions)."""
    # B, T = ids.shape
    # target_n = int(mask_frac * T)
    # mask_positions = np.zeros_like(ids, dtype=bool)
    # for b in range(B):
    #     masked = 0
    #     while masked < target_n:
    #         length = max(1, int(rng.geometric(1.0 / mean_span)))
    #         length = min(length, target_n - masked)
    #         start  = int(rng.integers(0, max(1, T - length)))
    #         if mask_positions[b, start:start + length].any():
    #             continue
    #         mask_positions[b, start:start + length] = True
    #         masked += length
    # corrupted = ids.copy()
    # corrupted[mask_positions] = mask_id
    # return corrupted, mask_positions
    # YOUR CODE HERE
    pass


check_geometric_span_mask(geometric_span_mask)


### Same window, two masking strategies

In [ ]:
def _single_demo(batch, rng):
    return mask_tokens(batch, MASK_ID, vocab_size, frac=0.20, rng=rng)

def _span_demo(batch, rng):
    return geometric_span_mask(batch, MASK_ID, mean_span=3.0,
                                mask_frac=0.20, rng=rng)

if (mask_tokens(ids[:16][None, :], MASK_ID, vocab_size, 0.15,
                np.random.default_rng(0)) is not None
    and geometric_span_mask(ids[:16][None, :], MASK_ID, 3.0, 0.20,
                             np.random.default_rng(0)) is not None):
    show_span_vs_single(ids, itos, _single_demo, _span_demo,
                        context=DEFAULT_CONTEXT, seed=4)
else:
    print('⬜ Need both mask_tokens and geometric_span_mask implemented.')


### Train with span masking

In [ ]:
params_sp = init_params(vocab_size, seed=1)

def _mask_span_strategy(batch, rng):
    return geometric_span_mask(batch, MASK_ID, mean_span=3.0,
                                mask_frac=0.15, rng=rng)

if (geometric_span_mask(ids[:16][None, :], MASK_ID, 3.0, 0.15,
                         np.random.default_rng(0)) is not None
    and masked_token_loss(np.zeros((1, 1, vocab_size), dtype=np.float32),
                          np.zeros((1, 1), dtype=np.int64),
                          np.array([[True]])) is not None):
    params_sp, losses_sp = train_lm(
        params_sp, ids, attention,
        mask_strategy_fn=_mask_span_strategy,
        use_causal_mask=False,
        n_steps=30, batch_size=32, seed=1,
    )
    plot_loss_curve(losses_sp, 'Span-masked LM training',
                    baseline=float(np.log(vocab_size)))
else:
    print('⬜ Need geometric_span_mask and masked_token_loss implemented first.')


**Observe:**
- The span-masked loss starts higher than the single-token loss at the same training step — the task is genuinely harder, exactly because the model cannot lean on local interpolation.
- After the same number of optimisation steps the span-masked model has a higher per-position loss but typically learns features that transfer better, the headline finding of SpanBERT and T5.
- We are still at 5k parameters and 40 L-BFGS-B steps; the difference is barely visible in the loss numbers but is the same mechanism that, at GPT-3 scale, makes span-corruption a default in encoder-decoder pretraining.


## 📜 Causal Next-Token Prediction (GPT)

Decoder-only language models take the same denoising principle and apply it strictly left to right. At every position, the model sees only the prefix and predicts the next token. The mask is no longer "predict the holes" but "you can never look at the future". The supervision signal is everywhere: at each of `T` positions in the sequence, the next character is the label.

> The masked LM trained on a small fraction of the tokens (15%) at each step. The causal LM trains on EVERY position. Why is the causal LM not therefore strictly better?

<details><summary>Thought</summary>

Two reasons, both about what the model is asked to predict. The masked LM at any masked position can use both left and right context, which is a richer signal per prediction; the causal LM at position `i` can only use the prefix `0..i-1`. The causal model has more predictions per pass, but each one is harder. Whether that trade-off favours one or the other depends on the downstream task: if you want to *generate* text, you need a left-to-right model; if you want to *understand* text (classification, retrieval, fill-in), the bidirectional one tends to be a better representation per parameter. The two recipes have therefore split into two architectural families.
</details>


### The causal mask

The causal mask is an additive `(T, T)` matrix: `0` for cells where the query is allowed to attend to the key (`j ≤ i`), and `-inf` for cells in the future (`j > i`). Adding this to the dot-product scores before the softmax sets the future weights to zero exactly.

Implement `causal_mask(T)`.

Useful: `np.full((T, T), -inf)`, `np.triu(..., k=1)` keeps the strict upper triangle and zeros the rest, then negate.


In [ ]:
def causal_mask(T):
    """Return (T, T) additive mask: 0 where j <= i, -inf where j > i."""
    # mask = np.full((T, T), -np.inf, dtype=np.float32)
    # mask = np.triu(mask, k=1)        # strict upper triangle stays -inf
    # mask = np.where(np.isneginf(mask), -np.inf, 0.0)
    # OR equivalently, start from zeros and set the strict upper tri to -inf.
    # YOUR CODE HERE
    pass


check_causal_mask(causal_mask)


### Picture: bidirectional vs causal

In [ ]:
if causal_mask(4) is not None:
    plot_attention_masks(context=8)
else:
    print('⬜ Need causal_mask implemented to render the diagram.')


### Next-token cross-entropy

At every position `t`, the model predicts position `t+1`. The loss is the cross-entropy of the predicted distribution against the actual next token, averaged over all but the last position (which has no follower to predict).

Implement `next_token_loss(logits, targets)` where `logits` has shape `(B, T, V)` and `targets` has shape `(B, T)`. The prediction at position `t` corresponds to `targets[t+1]`, so use `logits[:, :-1, :]` against `targets[:, 1:]`.


In [ ]:
def next_token_loss(logits, targets):
    """Cross-entropy of next-token prediction over a window."""
    # 1. pred_logits = logits[:, :-1, :]                # (B, T-1, V)
    # 2. pred_targets = targets[:, 1:]                  # (B, T-1)
    # 3. log_probs = stable_log_softmax(pred_logits)
    # 4. return -mean(log_probs[..., pred_targets]) at correct positions
    # YOUR CODE HERE
    pass


check_next_token_loss(next_token_loss)


### Train the causal LM

Same architecture as 🎭, with the causal mask switched on and the loss switched to `next_token_loss`. The helper handles both.


In [ ]:
params_cs = init_params(vocab_size, seed=2)

if (causal_mask(4) is not None
    and next_token_loss(np.zeros((1, 2, vocab_size), dtype=np.float32),
                        np.zeros((1, 2), dtype=np.int64)) is not None):
    params_cs, losses_cs = train_lm(
        params_cs, ids, attention,
        mask_strategy_fn=None,           # causal: predict every shifted position
        use_causal_mask=True,
        n_steps=30, batch_size=32, seed=2,
    )
    plot_loss_curve(losses_cs, 'Causal LM training',
                    baseline=float(np.log(vocab_size)))
else:
    print('⬜ Need causal_mask and next_token_loss implemented first.')


### Sampling: temperature controls how confident the model behaves

A trained causal LM defines a probability distribution over the next token. *Temperature* sampling rescales the logits by a constant `T` before the softmax. Low temperature concentrates probability mass on the top candidates; high temperature flattens the distribution and lets less-likely tokens win.

Implement `temperature_sample(logits, temperature, rng)`. Returns a single integer token id sampled from the rescaled distribution.


In [ ]:
def temperature_sample(logits, temperature, rng):
    """Sample one token id from the categorical distribution defined by logits / temperature."""
    # 1. scaled = logits / max(temperature, 1e-6)
    # 2. probs  = softmax(scaled)
    # 3. return int(rng.choice(len(probs), p=probs))
    # YOUR CODE HERE
    pass


check_temperature_sample(temperature_sample)


### Sample some Schiller-flavoured continuations

In [ ]:
if (causal_mask(4) is not None
    and temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                           np.random.default_rng(0)) is not None):
    seed_text = 'Karl Moor.'
    seed_ids  = encode_text(seed_text, stoi)
    out_ids = sample(params_cs, seed_ids, attention,
                     max_new_tokens=120, temperature=0.7, seed=0)
    print(f'PROMPT     : {seed_text!r}')
    print(f'CONTINUE   : {decode_ids(out_ids[len(seed_ids):], itos)!r}')
    print()

    out_ids2 = sample(params_cs, seed_ids, attention,
                      max_new_tokens=120, temperature=1.4, seed=0)
    print(f'TEMP=1.4   : {decode_ids(out_ids2[len(seed_ids):], itos)!r}')
else:
    print('⬜ Need causal_mask and temperature_sample implemented first.')


**Observe:**
- At low temperature (≈0.7) the continuation copies short Schiller-style fragments — repeated bigrams, common openings, capitalisation that follows the play's structure. The 5k-parameter model has not learned grammar; it has learned *texture*.
- At high temperature (≈1.4) the entropy of each step rises, the output drifts away from German into a broken letter soup; this is the "creativity vs coherence" knob you have heard about, in microcosm.
- A real GPT-style model is the same loss, the same causal mask, the same sampler — scaled up by six orders of magnitude in parameters, training data, and context. The pretext task is identical.

**At scale.** `mrkschtr/tiny_schiller` is a small GPT-style model trained on the same corpus we just used, with a subword tokeniser and a few transformer blocks. Loading it would require Hugging Face `transformers` plus PyTorch, neither of which fit in this kernel; the *model card* on Hugging Face shows the architecture and the loss curve, and running it locally via Ollama is the recommended extension (same recipe as chapter 8's "run a real foundation model on your laptop").


## 🗣️ Prompt Injection with Schiller

Once the model has been trained left-to-right, conditioning it on a prefix is the only steering tool we need. The prefix sits in front of the generated continuation and biases the distribution of every subsequent token toward whatever pattern the prefix establishes. At small scale this is *prompt steering*: a name like *Karl Moor.* makes the model continue in dialogue style. At large scale, the same mechanic becomes *prompt injection* — a deliberately crafted prefix can override the developer's intended behaviour and cause a system to follow whatever instruction the attacker put in front of it.

> The model was trained to predict the next character given a prefix. Why is "follow the instruction in the prompt" the same operation as "complete the prefix"?

<details><summary>Thought</summary>

Because the model has no separate channel for instructions: *everything* it sees is the prefix. If the training data contained patterns where an instruction-like phrase was followed by something that looked like compliance (which web-scale corpora do, abundantly), then continuing those patterns at inference time IS following an instruction. The model is not "deciding" whether to obey; it is sampling the most probable continuation, and instruction-following is the highest-probability continuation in many contexts. This is also why prompt injection is hard to fix with prompts alone — to the model, the malicious instruction and the developer's instruction have the same shape.
</details>


### Steering by name

The simplest form of conditioning. Three different character-name prefixes; the same model; three different distributions of continuations.

The 5k-parameter model you trained in the browser is too small to actually follow steering — it has barely learned the unigram statistics, let alone "what comes after a character name". The cell below loads the offline-trained version (d_model=32, 1500 Adam steps on the full corpus) automatically if available, falling back to your in-browser model otherwise.


In [ ]:
# Prefer the offline-trained model for the steering demos — it actually has
# enough capacity to condition on the prefix. Falls back to the in-browser
# model with a notice if the precomputed file is missing.
demo_params = load_precomputed_weights('causal_lm_weights.npz')
if demo_params is None:
    print('  ↳ no precomputed weights found, using in-browser params_cs.')
    print('     Run generate_precomputed.py for the full demo.')
    demo_params = params_cs
else:
    meta = precomputed_meta('causal_lm_weights.npz')
    print(f'  ↳ using precomputed weights  d_model={meta["d_model"]}  '
          f'context={meta["context"]}  (1500 Adam steps offline)')

if temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                       np.random.default_rng(0)) is not None:
    for prefix in ('Karl Moor.', 'Franz.', 'Amalia.', 'Räuber. '):
        seed_ids = encode_text(prefix, stoi)
        out_ids  = sample(demo_params, seed_ids, attention,
                          max_new_tokens=120, temperature=0.7, seed=0)
        cont = decode_ids(out_ids[len(seed_ids):], itos)
        print(f'{prefix:>14}  ->  {cont!r}')
else:
    print('⬜ Need temperature_sample.')


### Injecting an instruction

Prompt injection is the trick of embedding *what looks like new instructions* into a prompt, hoping the model treats them as authoritative. Our 5k-parameter character model is far too small for this to work for real — it has no concept of *instruction*, only of *pattern*. But we can show the mechanism by injecting a *style change* into the middle of a generation: continue from a Karl-Moor monologue, then mid-stream insert a Franz prefix and watch the style flip.


In [ ]:
# Two-stage generation: continue under one prefix, then graft a different prefix
# in the middle and resume. The model has no idea anything 'switched'; it just
# sees a prefix and keeps going.
if temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                       np.random.default_rng(0)) is not None:
    stage1 = encode_text('Karl Moor.', stoi)
    out1   = sample(demo_params, stage1, attention,
                    max_new_tokens=80, temperature=0.7, seed=0)
    print('STAGE 1  Karl-Moor monologue:')
    print(repr(decode_ids(out1, itos)))
    print()

    # Now inject a Franz prefix into the running context and resume.
    injection = encode_text('  Franz.', stoi)
    stage2 = np.concatenate([out1, injection])
    out2   = sample(demo_params, stage2, attention,
                    max_new_tokens=80, temperature=0.7, seed=0)
    print('STAGE 2  After injecting a Franz turn:')
    print(repr(decode_ids(out2[len(stage1):], itos)))
else:
    print('⬜ Need temperature_sample.')


**Observe:**
- The model never *knew* that the original instruction (continue Karl Moor) was authoritative. As soon as a different name prefix appears, it conditions on the new prefix and the style changes.
- This is the toy version of a real attack: at production scale, a model that retrieves text from the web, or accepts user-provided document fragments, can be steered by *anything* in those texts, not only by the developer's system prompt. Instruction-following is conditional generation, and conditional generation has no concept of "trusted source".
- Mitigations at scale include input/output filtering, separate channels for system vs user prompts (with extensive RLHF training), and tool-use sandboxes. None of them are bulletproof. The chapter's *"in the beginning was the word"* margin note is exactly this: prompts are the interface, and the interface is the attack surface.

**At scale.** Try this with `mrkschtr/tiny_schiller` (or any locally-runnable model via Ollama, e.g., `llama3.2:1b`) once outside this notebook. A few-billion-parameter model will *actually* follow injected instructions, and you can feel the failure mode directly in a single afternoon's worth of API calls.


### 🏁 Recap

**What we did:**
- 🔡 Loaded *Die Räuber* and built a char-level vocabulary, the simplest possible tokeniser. Every character mapped to an integer id; slot 0 reserved for a `[MASK]` sentinel.
- 🎭 Implemented bidirectional scaled dot-product attention, BERT-style 80/10/10 masking, and the masked-token cross-entropy. Trained a tiny encoder LM in the browser via L-BFGS-B and watched it fill in held-out windows.
- 🧩 Generalised single-token masking to *span* masking (geometric-length contiguous chunks). Same architecture, harder pretext, longer-range context required to solve it.
- 📜 Switched the attention pattern to causal (with the additive `(T, T)` upper-triangle mask), the loss to next-token cross-entropy, and the inference path to autoregressive sampling. Same architecture, same trainer, decoder-only model.
- 🗣️ Used the trained causal model to demonstrate *prompt steering* — the same mechanism that, scaled up, becomes prompt injection.

**Key takeaways:**
- Self-supervision turns the *structure of the data* into the supervision signal. We never wrote down a label; the masking strategy was the label.
- Encoder-only (bidirectional), decoder-only (causal), and encoder-decoder are the three architectural recipes built on the same denoising objective. The masking pattern decides which one you are training.
- Span masking forces longer-range integration than per-token masking at the same mask fraction. The harder pretext usually transfers better.
- Generation is conditional sampling. Conditional sampling has no idea what a "trusted prompt" is, which is why prompt injection is *not* a bug — it is the system working as designed.

The next chapter, semi-supervised and self-training methods, will close the loop with another way of squeezing supervision out of unlabelled data: letting the model label *itself*.


## Take It from Here, Next Steps

An optional extension that deepens the chapter without adding new infrastructure. Work through it at your own pace after the session.

### 🪞 JEPA — predict in latent space

Reconstructing tokens (or pixels) wastes capacity on noise the downstream task does not care about. *Joint-embedding predictive architectures* move the prediction target from raw input to a learned representation: a teacher network encodes the unmasked input, a student network sees the masked input and predicts the teacher's representations of the masked region in feature space. The loss is MSE between predicted and target features. There is no decoder.

Two structural pieces hold the method up:

- The **teacher and student share the same encoder architecture**, but the teacher's weights lag the student's by an exponential moving average. The teacher is what the student is *becoming*, slowly.
- The **target is in feature space**, not pixel/token space. The teacher's hidden state at a masked position is what the student tries to reproduce.

Implementing this in our setting is two short functions. With them in hand, you can train a JEPA student on the same Schiller corpus (re-using your `attention` and `mask_tokens`), and watch — via the embedding-variance plot — what would happen if you removed the EMA update (the answer: representation collapse, the chapter's open theoretical question).


### `ema_update`

In [ ]:
def ema_update(teacher_params, student_params, m=0.99):
    """Per-key EMA: teacher = m * teacher + (1 - m) * student."""
    # YOUR CODE HERE
    pass


check_ema_update(ema_update)


### `jepa_loss`

In [ ]:
def jepa_loss(student_h, teacher_h, mask_pos):
    """MSE between student and teacher hidden states at masked positions only."""
    # YOUR CODE HERE
    pass


check_jepa_loss(jepa_loss)


### What to try

A full training loop that uses these two helpers, reuses your `attention`, `mask_tokens`, and a no-mask `forward` for the teacher, and runs ~30 EMA steps on the same corpus. Track:

1. **Embedding variance per step** across the batch — should stay healthy (non-zero) because of the EMA delay between teacher and student.
2. **Ablation:** remove the stop-gradient (let the loss flow into the teacher too), or remove the EMA delay (set `m = 0`, copying the student verbatim). In either case, watch the variance fall toward zero — the canonical collapse diagnostic.

The same pattern, on images, is I-JEPA; on video, it is V-JEPA. Modality changes; the recipe does not.
